# ❤️ Heart Disease Prediction using Machine Learning

---

**J.N.N College of Engineering, Shivamogga**  
**Department of Computer Science and Engineering**  
**Subject: Machine Learning (BCS602) | Semester: 6**  
**Faculty: Mrs. Radhika S K**

---

## 📌 Problem Statement

Cardiovascular disease (CVD) is the **#1 cause of death globally**, taking an estimated **17.9 million lives each year** (WHO, 2023).  
Early and accurate prediction of heart disease can significantly reduce mortality rates by enabling timely medical intervention.

**Goal:** Build and compare machine learning models that can predict whether a patient has heart disease based on clinical measurements — helping clinicians make faster, data-driven decisions.

**Type:** Binary Classification (0 = No Disease, 1 = Heart Disease Present)

---

## 📊 Dataset

- **Name:** Cleveland Heart Disease Dataset  
- **Source:** UCI Machine Learning Repository (https://archive.ics.uci.edu/ml/datasets/heart+disease)  
- **Rows:** 303 patients | **Features:** 13 clinical attributes  
- **Target:** `target` (1 = Disease, 0 = No Disease)

| Feature | Description |
|---------|-------------|
| age | Age in years |
| sex | Sex (1=male, 0=female) |
| cp | Chest pain type (0-3) |
| trestbps | Resting blood pressure (mm Hg) |
| chol | Serum cholesterol (mg/dl) |
| fbs | Fasting blood sugar > 120 mg/dl (1=true) |
| restecg | Resting ECG results (0-2) |
| thalach | Maximum heart rate achieved |
| exang | Exercise-induced angina (1=yes) |
| oldpeak | ST depression induced by exercise |
| slope | Slope of peak exercise ST segment |
| ca | Number of major vessels colored by fluoroscopy |
| thal | Thalassemia (1=normal, 2=fixed defect, 3=reversible defect) |

---

## 🗂️ Project Workflow

```
Week 1-2: Problem Selection & Data Exploration (EDA + Cleaning)
Week 3:   Model Selection & Training (Logistic Regression, Random Forest, XGBoost)
Week 4:   Model Evaluation & Insights (Accuracy, Precision, Recall, F1, ROC-AUC)
Final:    Report & Submission
```

---
# 📦 Step 0: Install Required Libraries

In [ ]:
# Install required libraries (run once)
import subprocess, sys

packages = [
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'scikit-learn', 'xgboost', 'shap', 'plotly'
]
for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All libraries ready!')

---
# 📚 Step 1: Import Libraries

In [ ]:
# ─── Core Libraries ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# ─── XGBoost ──────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ─── SHAP (Explainability) ────────────────────────────────────────────────────
import shap

# ─── Settings ─────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
SEED = 42
np.random.seed(SEED)

print('✅ Libraries imported successfully!')

---
# 📥 Step 2: Load Dataset

In [ ]:
# Load Cleveland Heart Disease Dataset from UCI (via a reliable mirror)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'

columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

try:
    df = pd.read_csv(url, names=columns, na_values='?')
    print('✅ Dataset loaded from UCI Repository')
except Exception:
    # Fallback: create dataset from embedded values (guaranteed to work offline)
    print('⚠️  Network unavailable — using embedded dataset')
    
    data_str = """63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
57,0,0,120,354,0,1,163,1,0.6,2,0,2,1
57,1,0,140,192,0,1,148,0,0.4,1,0,1,1
56,0,1,140,294,0,0,153,0,1.3,1,0,2,1
44,1,1,120,263,0,1,173,0,0.0,2,0,3,1
52,1,2,172,199,1,1,162,0,0.5,2,0,3,1
57,1,2,150,168,0,1,174,0,1.6,2,0,2,1
54,1,0,140,239,0,1,160,0,1.2,2,0,2,1
48,0,2,130,275,0,1,139,0,0.2,2,0,2,1
49,1,1,130,266,0,1,171,0,0.6,2,0,2,1
64,1,3,110,211,0,0,144,1,1.8,1,0,2,1
58,0,3,150,283,1,0,162,0,1.0,2,0,2,1
50,0,2,120,219,0,1,158,0,1.6,1,0,2,1
58,0,2,120,340,0,1,172,0,0.0,2,0,2,1
66,0,3,150,226,0,1,114,0,2.6,0,0,2,1
43,1,0,150,247,0,1,171,0,1.5,2,0,2,1
69,0,3,140,239,0,1,151,0,1.8,2,2,2,1
59,1,0,135,234,0,1,161,0,0.5,1,0,3,1
44,1,2,130,233,0,1,179,1,0.4,2,0,2,1
42,1,0,140,226,0,1,178,0,0.0,2,0,2,1
61,1,2,150,243,1,1,137,1,1.0,1,0,2,1
40,1,3,140,199,0,1,178,1,1.4,2,0,3,1
71,0,1,160,302,0,1,162,0,0.4,2,2,2,1
59,1,2,150,212,1,1,157,0,1.6,2,0,2,1
51,1,2,110,175,0,1,123,0,0.6,2,0,2,1
65,0,2,140,417,1,0,157,0,0.8,2,1,2,1
53,1,2,130,197,1,0,152,0,1.2,0,0,2,1
41,0,1,105,198,0,1,168,0,0.0,2,1,2,1
65,1,0,120,177,0,1,140,0,0.4,2,0,3,1
44,1,1,130,219,0,0,188,0,0.0,2,0,2,1
54,1,2,125,273,0,0,152,0,0.5,0,1,2,1
51,1,3,125,213,0,0,125,1,1.4,2,1,2,1
46,0,1,142,177,0,0,160,1,1.4,0,0,2,1
54,0,2,135,304,1,1,170,0,0.0,2,0,2,1
54,1,2,150,195,0,1,122,0,0.0,2,0,2,1
60,1,0,130,206,0,0,132,1,2.4,1,2,3,1
60,1,2,140,293,0,0,170,0,1.2,2,2,3,1
54,1,2,150,230,0,1,140,1,1.2,1,0,3,1
59,1,1,140,177,0,1,162,1,0.0,2,1,3,1
46,1,2,120,231,0,1,115,1,0.0,2,0,2,1
65,0,2,155,269,0,1,148,0,0.8,2,0,2,1
67,1,0,160,286,0,0,108,1,1.5,1,3,2,0
67,1,0,120,229,0,0,129,1,2.6,1,2,3,0
62,0,0,140,268,0,0,160,0,3.6,0,2,2,0
63,1,0,130,254,0,0,147,0,1.4,1,1,3,0
53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
56,1,2,130,256,1,0,142,1,0.6,1,1,1,0
48,1,1,110,229,0,1,168,0,1.0,0,0,3,0
58,1,1,120,284,0,0,160,0,1.8,1,0,2,0
58,1,2,132,224,0,0,173,0,3.2,2,2,3,0
60,1,0,130,206,0,0,132,1,2.4,1,2,3,0
50,0,0,110,254,0,0,159,0,0.0,2,0,2,0
53,1,3,130,246,1,0,173,0,0.0,2,3,2,0
45,0,1,130,234,0,0,175,0,0.6,1,0,2,0
65,1,0,110,248,0,0,158,0,0.6,2,2,1,0
69,1,3,160,234,1,0,131,0,0.1,1,1,2,0
69,0,3,140,239,0,1,151,0,1.8,2,2,2,0
67,1,0,100,299,0,0,125,1,0.9,1,2,3,0
68,0,2,120,211,0,0,115,0,1.5,1,0,2,0
34,1,2,118,182,0,0,174,0,0.0,2,0,2,1
62,0,3,140,394,0,0,157,0,1.2,1,0,2,1
51,1,2,140,261,0,0,186,1,0.0,2,0,2,1
52,1,2,128,255,0,1,161,1,0.0,2,1,3,0
46,1,0,150,231,0,1,147,0,3.6,1,0,2,0
54,1,0,150,195,0,1,122,0,0.0,2,0,2,0
58,0,0,100,248,0,0,122,0,1.0,1,0,2,0
71,0,2,110,265,1,0,130,0,0.0,2,1,3,0
57,1,0,150,276,0,0,112,1,0.6,1,1,1,0
71,0,0,112,149,0,1,125,0,1.6,1,0,2,0
45,1,2,142,309,0,0,147,1,0.0,1,3,3,0
62,1,2,120,267,0,1,99,1,1.8,1,2,3,0
59,1,0,164,176,1,0,90,0,1.0,1,2,1,0
51,0,2,130,305,0,1,142,1,1.2,1,0,3,0
44,1,1,120,220,0,1,170,0,0.0,2,0,2,1
60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
44,1,0,112,290,0,0,153,0,0.0,2,1,2,0
42,1,2,130,180,0,1,150,0,0.0,2,0,2,1
61,1,2,150,243,1,1,137,1,1.0,1,0,2,0
66,0,0,178,228,1,1,165,1,1.0,1,2,3,0
46,1,0,110,240,0,1,140,0,0.0,2,0,3,0
71,0,2,160,302,0,1,162,0,0.4,2,2,2,0
64,1,3,110,211,0,0,144,1,1.8,1,0,2,0
40,1,2,152,223,0,1,181,0,0.0,2,0,3,1
57,1,2,150,126,1,1,173,0,0.2,2,1,3,1
58,1,0,146,218,0,1,105,0,2.0,1,1,3,0
52,1,0,112,230,0,1,160,0,0.0,2,1,2,1
54,0,2,108,267,0,0,167,0,0.0,2,0,2,1
60,1,0,130,253,0,1,144,1,1.4,2,1,3,0
56,1,0,132,184,0,0,105,1,2.1,1,1,1,0
62,0,3,140,394,0,0,157,0,1.2,1,0,2,0
54,1,0,124,266,0,0,109,1,2.2,1,1,3,0
38,1,2,138,175,0,1,173,0,0.0,2,4,2,1
43,1,0,150,247,0,1,171,0,1.5,2,0,2,0
58,0,3,150,283,1,0,162,0,1.0,2,0,2,1
29,1,1,130,204,0,0,202,0,0.0,2,0,2,1
62,0,3,160,164,0,0,145,0,6.2,0,3,3,0
53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
74,0,1,120,269,0,0,121,1,0.2,2,1,2,0
56,1,0,130,256,1,0,142,1,0.6,1,1,1,0
46,1,0,120,249,0,0,144,0,0.8,2,0,3,0
44,1,2,140,235,0,0,180,0,0.0,2,0,2,1
41,1,2,110,235,0,1,153,0,0.0,2,0,2,1
48,0,2,130,275,0,1,139,0,0.2,2,0,2,1
48,1,1,130,245,0,0,160,0,0.0,2,0,2,1
52,1,2,172,199,1,1,162,0,0.5,2,0,3,1
56,1,2,150,213,0,0,125,1,1.0,1,2,2,0
57,1,0,150,255,0,1,92,1,3.0,1,2,2,0
64,1,0,120,246,0,0,96,1,2.2,0,1,2,0
66,1,2,160,228,0,0,138,0,2.3,2,0,1,0
63,1,3,145,233,1,0,150,0,2.3,0,0,1,0
54,1,0,130,242,0,0,91,1,1.0,1,0,3,0
47,1,0,138,257,0,0,156,0,0.0,2,0,2,0
57,1,1,140,265,0,1,145,1,1.0,1,2,2,0
64,0,3,140,313,0,1,133,0,0.2,2,0,3,1
58,0,3,150,283,1,0,162,0,1.0,2,0,2,1
57,0,0,120,354,0,1,163,1,0.6,2,0,2,1
70,1,2,156,245,0,0,143,0,0.0,2,0,2,1
46,1,2,140,311,0,1,120,1,1.8,1,2,3,0
54,1,0,122,286,0,0,116,1,3.2,1,2,2,0
71,0,0,112,149,0,1,125,0,1.6,1,0,2,0
43,0,2,122,213,0,1,165,0,0.2,1,0,2,1
34,0,1,118,210,0,1,192,0,0.7,2,0,2,1
35,1,1,122,192,0,1,174,0,0.0,2,0,2,1
51,1,2,128,213,0,0,125,1,1.2,1,1,2,0
45,0,1,112,160,0,1,138,0,0.0,1,0,2,1
38,0,2,138,175,0,1,173,0,0.0,2,4,2,1
56,0,1,140,294,0,0,153,0,1.3,1,0,2,1
49,0,2,134,271,0,1,162,0,0.0,1,0,2,1
54,1,2,192,283,0,0,195,0,0.0,2,1,3,0
59,1,3,134,204,0,1,162,0,0.8,2,2,2,1
57,1,0,154,232,0,0,164,0,0.0,2,1,2,1
61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
47,1,2,108,243,0,1,152,0,0.0,2,0,2,1
53,0,2,107,250,0,0,154,0,0.0,2,0,3,1
65,0,2,160,360,0,0,151,0,0.8,2,0,2,1
64,0,2,145,212,0,0,132,0,2.0,1,2,1,0
44,1,1,130,233,0,1,179,1,0.4,2,0,2,1
68,1,2,118,277,0,1,151,0,1.0,2,1,3,0
55,1,0,140,217,0,1,111,1,5.6,0,0,3,0
57,1,2,145,191,0,1,132,0,0.1,1,0,3,1
36,1,1,120,166,0,1,180,0,0.0,2,0,2,1
62,1,0,140,294,0,0,153,0,1.3,1,0,2,0
60,0,2,150,258,0,0,157,0,2.6,1,2,3,0
63,0,2,135,252,0,0,172,0,0.0,2,0,2,1
57,1,2,150,126,1,1,173,0,0.2,2,1,3,1
55,1,2,132,353,0,1,132,1,1.2,1,1,3,0
46,1,0,150,231,0,1,147,0,3.6,1,0,2,0
43,0,2,122,213,0,1,165,0,0.2,1,0,2,1
52,1,1,134,201,0,1,158,0,0.8,2,1,2,1
41,1,1,135,203,0,1,132,0,0.0,1,0,1,1
58,1,2,140,211,1,0,165,0,0.0,2,0,2,1
35,0,0,138,183,0,1,182,0,1.4,2,0,2,1
58,0,0,100,248,0,0,122,0,1.0,1,0,2,0
52,1,0,112,230,0,1,160,0,0.0,2,1,2,1
57,1,0,110,335,0,1,143,1,3.0,1,1,3,0
62,1,2,120,267,0,1,99,1,1.8,1,2,3,0
66,1,0,120,302,0,0,151,0,0.4,1,0,2,0
41,1,2,112,250,0,1,179,0,0.0,2,0,2,1
42,1,0,136,315,0,1,125,1,1.8,1,0,1,0
59,1,2,126,218,1,1,134,0,2.2,1,1,1,0
50,1,2,129,196,0,1,163,0,0.0,2,0,2,1
44,0,2,118,242,0,1,149,0,0.3,1,1,2,0
58,0,2,136,319,1,0,152,0,0.0,2,2,2,0
54,0,1,160,201,0,1,163,0,0.0,2,1,2,1
56,1,1,120,240,0,1,169,0,0.0,0,0,2,0
46,1,0,120,230,0,1,150,0,0.0,1,0,2,1
49,0,2,134,271,0,1,162,0,0.0,1,0,2,1
56,0,2,134,409,0,0,150,1,1.9,1,2,3,0
62,1,0,160,254,0,0,108,1,3.0,1,2,3,0
62,0,0,140,394,0,0,157,0,1.2,1,0,2,1
52,0,0,136,196,0,0,169,0,0.1,1,0,2,1
55,1,0,160,289,0,0,145,1,0.8,1,1,3,0
51,1,2,100,222,0,1,143,1,1.2,1,0,2,0
48,1,1,124,255,1,1,175,0,0.0,2,2,2,0
44,0,2,102,318,0,1,160,0,0.0,2,1,2,1
63,1,0,130,254,0,0,147,0,1.4,1,1,3,0
76,0,2,140,197,0,0,116,0,1.1,1,0,2,1
54,1,0,122,286,0,0,116,1,3.2,1,2,2,0
50,1,0,144,200,0,0,126,1,0.9,1,0,3,0
64,1,0,128,263,0,1,105,1,0.2,1,1,3,0
70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
57,1,1,140,265,0,1,145,1,1.0,1,2,2,0
58,1,0,146,218,0,1,105,0,2.0,1,1,3,0
48,1,2,130,256,1,0,150,1,0.0,2,2,3,0
66,1,2,160,228,0,0,138,0,2.3,2,0,1,0
60,1,0,130,253,0,1,144,1,1.4,2,1,3,0
60,1,0,140,293,0,0,170,0,1.2,2,2,3,0
56,1,0,150,213,1,0,125,1,1.0,1,2,2,0
55,1,0,132,342,0,1,166,0,1.2,2,0,2,0
70,1,0,160,269,0,1,112,1,2.9,1,1,3,0
63,0,0,150,407,0,0,154,0,4.0,1,3,3,0
52,1,0,128,255,0,1,161,1,0.0,2,1,3,0
45,1,2,104,208,0,0,148,1,3.0,1,0,2,0
63,0,3,135,252,0,0,172,0,0.0,2,0,2,1
59,1,0,110,239,0,0,142,1,1.2,1,1,3,0
52,1,3,152,298,1,1,178,0,1.2,1,0,3,0
57,1,0,150,255,0,1,92,1,3.0,1,2,2,0
51,1,3,125,188,0,1,145,0,0.0,2,0,2,1
52,1,0,122,212,0,0,84,0,0.0,2,0,2,0
53,1,3,130,246,1,0,173,0,0.0,2,3,2,0
66,1,0,112,212,0,0,132,1,0.1,2,1,2,0
68,1,2,118,277,0,1,151,0,1.0,2,1,3,0
68,0,2,120,211,0,0,115,0,1.5,1,0,2,0
60,1,2,140,293,0,0,170,0,1.2,2,2,3,0
62,1,2,130,231,0,1,146,0,1.8,1,3,3,0
71,0,2,110,265,1,0,130,0,0.0,2,1,3,0
63,1,0,140,195,0,1,179,0,0.0,2,2,2,0
55,1,1,130,262,0,1,155,0,0.0,2,0,2,1
66,0,0,178,228,1,1,165,1,1.0,1,2,3,0
41,1,0,110,172,0,0,158,0,0.0,2,0,3,0
52,1,1,152,298,1,1,178,0,1.2,1,0,3,0
64,0,0,130,303,0,1,122,0,2.0,1,2,2,0
44,0,2,118,242,0,1,149,0,0.3,1,1,2,0
55,0,1,132,342,0,1,166,0,1.2,2,0,2,0
47,1,2,110,275,0,0,118,1,1.0,1,1,2,0
58,0,2,136,319,1,0,152,0,0.0,2,2,2,0
52,1,2,134,201,0,1,158,0,0.8,2,1,2,1
62,0,0,130,263,0,1,97,0,1.2,1,1,3,0
56,1,0,132,184,0,0,105,1,2.1,1,1,1,0
51,1,2,110,175,0,1,123,0,0.6,2,0,2,1
63,1,0,130,330,1,0,132,1,1.8,2,3,3,0
54,1,0,130,242,0,0,91,1,1.0,1,0,3,0
55,1,2,130,262,0,1,155,0,0.0,2,0,2,1
55,1,3,132,353,0,1,132,1,1.2,1,1,3,0
56,1,2,128,208,1,0,140,0,0.0,2,0,3,0
59,1,0,136,239,0,0,142,1,1.8,1,2,3,0
51,1,3,100,222,0,1,143,1,1.2,1,0,2,0
60,0,2,102,318,0,1,160,0,0.0,2,1,2,1
52,1,1,130,254,0,0,147,0,1.4,1,1,3,0
61,1,0,148,203,0,1,161,0,0.0,2,1,3,0"""

    from io import StringIO
    df = pd.read_csv(StringIO(data_str), names=columns)

# Convert target to binary (0=No Disease, 1=Disease)
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print(f'Dataset shape: {df.shape}')
print(f'Target distribution:\n{df["target"].value_counts().rename({0: "No Disease", 1: "Heart Disease"})}' )
df.head(10)

---
# 🔍 Step 3: Exploratory Data Analysis (EDA)

## 3.1 Basic Statistics

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Statistical Summary ===')
df.describe().round(2)

## 3.2 Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ No missing values found!')
else:
    print(missing_df)
    
    # Visualize missing values
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['Missing %'].plot(kind='bar', color='coral', ax=ax)
    ax.set_title('Missing Values by Feature (%)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Missing %')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3.3 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar Chart
counts = df['target'].value_counts()
bars = axes[0].bar(['No Heart Disease', 'Heart Disease'], counts.values,
                   color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=1.2)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Patients')
axes[0].set_ylim(0, max(counts.values) * 1.2)

# Pie Chart
axes[1].pie(counts.values, labels=['No Disease', 'Heart Disease'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, shadow=True, explode=(0.05, 0.05),
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Target Variable Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Heart Disease Dataset — Target Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n📌 Dataset is {"balanced" if abs(counts[0]-counts[1])/len(df) < 0.1 else "slightly imbalanced"}')
print(f'   No Disease: {counts[0]} patients ({counts[0]/len(df)*100:.1f}%)')
print(f'   Heart Disease: {counts[1]} patients ({counts[1]/len(df)*100:.1f}%)')

## 3.4 Feature Distributions by Target

In [ ]:
# Continuous features distribution
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors = {0: '#2ecc71', 1: '#e74c3c'}
labels = {0: 'No Disease', 1: 'Heart Disease'}

for idx, feat in enumerate(continuous_features):
    for target_val in [0, 1]:
        subset = df[df['target'] == target_val][feat]
        axes[idx].hist(subset, bins=20, alpha=0.6, color=colors[target_val],
                       label=labels[target_val], edgecolor='white')
    axes[idx].set_title(f'{feat.upper()} Distribution', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(feat)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

# Hide unused subplot
axes[-1].axis('off')

plt.suptitle('Continuous Feature Distributions by Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, feat in enumerate(categorical_features):
    ct = pd.crosstab(df[feat], df['target'], normalize='index') * 100
    ct.columns = ['No Disease %', 'Heart Disease %']
    ct.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'],
            edgecolor='black', linewidth=0.8)
    axes[idx].set_title(f'{feat.upper()}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Percentage (%)')
    axes[idx].set_xlabel('')
    axes[idx].legend(fontsize=8)
    axes[idx].tick_params(axis='x', rotation=0)

plt.suptitle('Categorical Features: Heart Disease % by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.5 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax,
            annot_kws={'size': 9})

ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Top correlations with target
print('\n📌 Top Features Correlated with Target (Heart Disease):')
target_corr = corr['target'].drop('target').abs().sort_values(ascending=False)
for feat, val in target_corr.items():
    direction = '🔴 Positive' if corr['target'][feat] > 0 else '🟢 Negative'
    print(f'   {feat:12s}: {val:.3f}  ({direction} correlation)')

## 3.6 Box Plots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6))

for idx, feat in enumerate(continuous_features):
    df.boxplot(column=feat, by='target', ax=axes[idx],
               boxprops=dict(color='navy'),
               medianprops=dict(color='red', linewidth=2),
               whiskerprops=dict(color='navy'),
               flierprops=dict(marker='o', color='orange', alpha=0.5))
    axes[idx].set_title(feat.upper(), fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('0=No Disease | 1=Heart Disease')
    axes[idx].set_xticklabels(['No Disease', 'Disease'], rotation=10)

plt.suptitle('Box Plots: Feature Spread by Target Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.7 Pairplot — Key Features

In [ ]:
key_features = ['age', 'thalach', 'oldpeak', 'chol', 'target']
pair_df = df[key_features].copy()
pair_df['target'] = pair_df['target'].map({0: 'No Disease', 1: 'Heart Disease'})

g = sns.pairplot(pair_df, hue='target', palette={'No Disease': '#2ecc71', 'Heart Disease': '#e74c3c'},
                 diag_kind='kde', plot_kws={'alpha': 0.6, 's': 40})
g.fig.suptitle('Pairplot of Key Clinical Features', y=1.02, fontsize=14, fontweight='bold')
plt.show()

---
# 🧹 Step 4: Data Preprocessing & Cleaning

## 4.1 Handle Missing Values

In [ ]:
print(f'Dataset shape before cleaning: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

# Fill missing values with median (robust to outliers)
for col in df.columns:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  ✅ Filled "{col}" missing values with median = {median_val}')

# Ensure correct dtypes
int_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'target']
for col in int_cols:
    df[col] = df[col].astype(int)

print(f'\nDataset shape after cleaning: {df.shape}')
print('✅ No more missing values:', df.isnull().sum().sum() == 0)

## 4.2 Outlier Detection & Treatment (IQR Method)

In [ ]:
print('=== Outlier Analysis (IQR Method) ===')
outlier_summary = {}

for col in continuous_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    outlier_summary[col] = {
        'count': len(outliers), 
        'pct': round(len(outliers)/len(df)*100, 2),
        'lower_bound': round(lower, 2),
        'upper_bound': round(upper, 2)
    }
    
    # Cap outliers (Winsorization) — preserve data, just clip extremes
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f'  {col:12s}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%) — capped at [{lower:.1f}, {upper:.1f}]')

print('\n✅ Outliers capped using Winsorization (1.5×IQR method)')

## 4.3 Feature Engineering

In [ ]:
# Create meaningful derived features

# Age Group
df['age_group'] = pd.cut(df['age'], bins=[0, 40, 50, 60, 100],
                          labels=['<40', '40-50', '50-60', '>60']).astype(int)

# Heart rate reserve indicator (low max HR is a risk factor)
df['hr_reserve'] = df['thalach'] - (220 - df['age'])

# Chest pain severity flag
df['has_chest_pain'] = (df['cp'] > 0).astype(int)

print('✅ New features added:')
print('   - age_group    : Age categorized into bins')
print('   - hr_reserve   : Heart rate reserve (actual - predicted max HR)')
print('   - has_chest_pain: Binary flag for any chest pain type')
print(f'\nUpdated dataset shape: {df.shape}')

## 4.4 Train-Test Split & Feature Scaling

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

feature_names = X.columns.tolist()
print(f'Features ({len(feature_names)}): {feature_names}')
print(f'Target shape: {y.shape}')

# Stratified train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f'\n✅ Train-Test Split (80/20 stratified):')
print(f'   Training set : {X_train.shape[0]} samples')
print(f'   Test set     : {X_test.shape[0]} samples')
print(f'   Train target: {dict(y_train.value_counts())}')
print(f'   Test target : {dict(y_test.value_counts())}')

# Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)

print('\n✅ Features scaled using StandardScaler')

---
# 🤖 Step 5: Model Training

## 5.1 Model 1 — Logistic Regression

> **Justification:** Logistic Regression is the classical baseline for binary classification. It is highly interpretable, computationally efficient, and provides probability estimates. Ideal as a medical baseline model where explainability matters.

In [ ]:
print('=' * 55)
print('        MODEL 1: LOGISTIC REGRESSION')
print('=' * 55)

# Hyperparameter tuning with GridSearchCV
lr_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [500]
}

lr_base = LogisticRegression(random_state=SEED)
lr_cv = GridSearchCV(lr_base, lr_params, cv=StratifiedKFold(n_splits=5),
                     scoring='roc_auc', n_jobs=-1, verbose=0)
lr_cv.fit(X_train_scaled, y_train)

lr_model = lr_cv.best_estimator_
print(f'\n📌 Best Parameters: {lr_cv.best_params_}')
print(f'   Best CV ROC-AUC : {lr_cv.best_score_:.4f}')

# Predictions
lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# Cross-validation scores
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train,
                                cv=StratifiedKFold(5), scoring='accuracy')

print(f'\n   5-Fold CV Accuracy: {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}')
print(f'\n   Test Set Metrics:')
print(f'   Accuracy   : {accuracy_score(y_test, lr_pred):.4f}')
print(f'   Precision  : {precision_score(y_test, lr_pred):.4f}')
print(f'   Recall     : {recall_score(y_test, lr_pred):.4f}')
print(f'   F1-Score   : {f1_score(y_test, lr_pred):.4f}')
print(f'   ROC-AUC    : {roc_auc_score(y_test, lr_prob):.4f}')

## 5.2 Model 2 — Random Forest Classifier

> **Justification:** Random Forest is an ensemble method that builds multiple decision trees and aggregates their predictions. It handles non-linear relationships, is robust to overfitting, provides built-in feature importance, and consistently outperforms simple models on tabular medical data.

In [ ]:
print('=' * 55)
print('      MODEL 2: RANDOM FOREST CLASSIFIER')
print('=' * 55)

# Hyperparameter tuning
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2']
}

rf_base = RandomForestClassifier(random_state=SEED)
rf_cv = GridSearchCV(rf_base, rf_params, cv=StratifiedKFold(n_splits=5),
                     scoring='roc_auc', n_jobs=-1, verbose=0)
rf_cv.fit(X_train_scaled, y_train)

rf_model = rf_cv.best_estimator_
print(f'\n📌 Best Parameters: {rf_cv.best_params_}')
print(f'   Best CV ROC-AUC : {rf_cv.best_score_:.4f}')

# Predictions
rf_pred = rf_model.predict(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

# Cross-validation scores
rf_cv_scores = cross_val_score(rf_model, X_train_scaled, y_train,
                                cv=StratifiedKFold(5), scoring='accuracy')

print(f'\n   5-Fold CV Accuracy: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')
print(f'\n   Test Set Metrics:')
print(f'   Accuracy   : {accuracy_score(y_test, rf_pred):.4f}')
print(f'   Precision  : {precision_score(y_test, rf_pred):.4f}')
print(f'   Recall     : {recall_score(y_test, rf_pred):.4f}')
print(f'   F1-Score   : {f1_score(y_test, rf_pred):.4f}')
print(f'   ROC-AUC    : {roc_auc_score(y_test, rf_prob):.4f}')

## 5.3 Model 3 — XGBoost (Bonus / State-of-the-Art)

> **Justification:** XGBoost (Extreme Gradient Boosting) is the industry-standard algorithm that wins most tabular data competitions. It combines gradient boosting with regularization techniques, handles missing values natively, and scales efficiently.

In [ ]:
print('=' * 55)
print('        MODEL 3: XGBoost CLASSIFIER')
print('=' * 55)

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(random_state=SEED, eval_metric='logloss', use_label_encoder=False)
xgb_cv = GridSearchCV(xgb_base, xgb_params, cv=StratifiedKFold(n_splits=5),
                      scoring='roc_auc', n_jobs=-1, verbose=0)
xgb_cv.fit(X_train_scaled, y_train)

xgb_model = xgb_cv.best_estimator_
print(f'\n📌 Best Parameters: {xgb_cv.best_params_}')
print(f'   Best CV ROC-AUC : {xgb_cv.best_score_:.4f}')

# Predictions
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Cross-validation scores
xgb_cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train,
                                 cv=StratifiedKFold(5), scoring='accuracy')

print(f'\n   5-Fold CV Accuracy: {xgb_cv_scores.mean():.4f} ± {xgb_cv_scores.std():.4f}')
print(f'\n   Test Set Metrics:')
print(f'   Accuracy   : {accuracy_score(y_test, xgb_pred):.4f}')
print(f'   Precision  : {precision_score(y_test, xgb_pred):.4f}')
print(f'   Recall     : {recall_score(y_test, xgb_pred):.4f}')
print(f'   F1-Score   : {f1_score(y_test, xgb_pred):.4f}')
print(f'   ROC-AUC    : {roc_auc_score(y_test, xgb_prob):.4f}')

---
# 📊 Step 6: Model Evaluation & Comparison

## 6.1 Comprehensive Metrics Table

In [ ]:
models_info = [
    ('Logistic Regression', lr_pred, lr_prob, '#3498db'),
    ('Random Forest',       rf_pred, rf_prob, '#e67e22'),
    ('XGBoost',             xgb_pred, xgb_prob, '#9b59b6'),
]

results = []
for name, pred, prob, _ in models_info:
    results.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, pred),  4),
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall':    round(recall_score(y_test, pred),    4),
        'F1-Score':  round(f1_score(y_test, pred),        4),
        'ROC-AUC':   round(roc_auc_score(y_test, prob),   4),
    })

results_df = pd.DataFrame(results).set_index('Model')

print('\n' + '='*65)
print('          COMPREHENSIVE MODEL COMPARISON TABLE')
print('='*65)
print(results_df.to_string())
print('='*65)

# Highlight best model
best_model_name = results_df['ROC-AUC'].idxmax()
print(f'\n🏆 Best Model by ROC-AUC: {best_model_name} ({results_df.loc[best_model_name, "ROC-AUC"]:.4f})')

# Style the table
results_df.style.highlight_max(axis=0, color='lightgreen')

## 6.2 Metrics Bar Chart Comparison

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_colors = ['#3498db', '#e67e22', '#9b59b6']
model_names = ['Logistic Regression', 'Random Forest', 'XGBoost']

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 7))

for i, (name, color) in enumerate(zip(model_names, model_colors)):
    vals = [results_df.loc[name, m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color,
                  edgecolor='black', linewidth=0.7, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xlabel('Metric', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Model Performance Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=11)
ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='0.90 target')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred, prob, color) in zip(axes, models_info):
    cm = confusion_matrix(y_test, pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Disease', 'Heart Disease'],
                yticklabels=['No Disease', 'Heart Disease'],
                linewidths=2, linecolor='white',
                annot_kws={'size': 14, 'weight': 'bold'})
    
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{name}\n'
                 f'TP={tp} | TN={tn} | FP={fp} | FN={fn}',
                 fontsize=10, fontweight='bold')
    ax.set_ylabel('Actual', fontsize=11)
    ax.set_xlabel('Predicted', fontsize=11)

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 Key:')
print('  TP = True Positive  (Correctly predicted disease)')
print('  TN = True Negative  (Correctly predicted no disease)')
print('  FP = False Positive (Predicted disease, actually healthy)')
print('  FN = False Negative (Missed disease — most critical!)')

## 6.4 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for (name, pred, prob, color) in models_info:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, lw=2.5, color=color,
            label=f'{name} (AUC = {auc:.4f})')

# Random classifier baseline
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.5000)')

ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=13)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=13)
ax.set_title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📌 AUC Interpretation:')
print('   0.90 - 1.00 → Excellent')
print('   0.80 - 0.90 → Good')
print('   0.70 - 0.80 → Fair')
print('   0.60 - 0.70 → Poor')

## 6.5 Classification Reports

In [ ]:
for name, pred, prob, _ in models_info:
    print(f'\n{"="*50}')
    print(f'  {name} — Classification Report')
    print(f'{"="*50}')
    print(classification_report(y_test, pred,
                                 target_names=['No Disease', 'Heart Disease']))

---
# 🔬 Step 7: Model Explainability with SHAP

> SHAP (SHapley Additive exPlanations) provides **global and local explainability** — showing WHICH features influence predictions and by HOW MUCH for each patient. This is critical for medical AI trustworthiness.

In [ ]:
# SHAP for Random Forest (best ensemble model)
print('🔍 Computing SHAP values for Random Forest...')

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_scaled)

# For binary classification, shap_values is a list [class_0, class_1]
if isinstance(shap_values, list):
    shap_vals_pos = shap_values[1]  # SHAP for class 1 (Heart Disease)
else:
    shap_vals_pos = shap_values

print('✅ SHAP values computed!')

In [ ]:
# Global Feature Importance — SHAP Summary Bar Plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals_pos, X_test_scaled,
                  plot_type='bar', show=False,
                  color='#e74c3c')
plt.title('SHAP Feature Importance (Random Forest) — Impact on Heart Disease Prediction',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Beeswarm Plot — Direction and Magnitude
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals_pos, X_test_scaled, show=False)
plt.title('SHAP Beeswarm Plot — Feature Contribution to Heart Disease Risk',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 How to read this plot:')
print('   Red  = High feature value | Blue = Low feature value')
print('   Right of center = Increases heart disease risk')
print('   Left of center  = Decreases heart disease risk')

In [ ]:
# SHAP Waterfall for a single patient (local explanation)
patient_idx = 0
actual = y_test.iloc[patient_idx]
predicted = rf_pred[patient_idx]

print(f'\n📋 Local Explanation for Test Patient #{patient_idx}')
print(f'   Actual:    {"Heart Disease" if actual == 1 else "No Disease"}')
print(f'   Predicted: {"Heart Disease" if predicted == 1 else "No Disease"}')
print(f'   Probability of Disease: {rf_prob[patient_idx]*100:.1f}%')

# SHAP waterfall plot
shap_explanation = shap.Explanation(
    values=shap_vals_pos[patient_idx],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    data=X_test_scaled.iloc[patient_idx].values,
    feature_names=feature_names
)

plt.figure(figsize=(12, 6))
shap.waterfall_plot(shap_explanation, show=False, max_display=15)
plt.title(f'SHAP Waterfall: Patient #{patient_idx} — Individual Prediction Explanation',
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 🎯 Step 8: Feature Importance Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Random Forest Feature Importance
rf_importance = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=True)
colors_rf = ['#e74c3c' if v > rf_importance.median() else '#3498db' for v in rf_importance.values]
rf_importance.plot(kind='barh', ax=axes[0], color=colors_rf, edgecolor='black', linewidth=0.5)
axes[0].set_title('Random Forest\nFeature Importances (Gini)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance Score')
axes[0].axvline(x=rf_importance.median(), color='gray', linestyle='--', alpha=0.7)

# XGBoost Feature Importance
xgb_importance = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=True)
colors_xgb = ['#e74c3c' if v > xgb_importance.median() else '#9b59b6' for v in xgb_importance.values]
xgb_importance.plot(kind='barh', ax=axes[1], color=colors_xgb, edgecolor='black', linewidth=0.5)
axes[1].set_title('XGBoost\nFeature Importances (Gain)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance Score')
axes[1].axvline(x=xgb_importance.median(), color='gray', linestyle='--', alpha=0.7)

plt.suptitle('Feature Importance Analysis — Ensemble Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top 5 features consensus
print('\n📌 Top 5 Most Important Features:')
rf_top5 = set(rf_importance.nlargest(5).index)
xgb_top5 = set(xgb_importance.nlargest(5).index)
consensus = rf_top5.intersection(xgb_top5)
print(f'   Random Forest Top 5: {list(rf_importance.nlargest(5).index)}')
print(f'   XGBoost Top 5:       {list(xgb_importance.nlargest(5).index)}')
print(f'   Consensus (both):    {list(consensus)}')

---
# 🩺 Step 9: Interactive Prediction Demo

In [ ]:
def predict_heart_disease(patient_data: dict, model=rf_model):
    """
    Predict heart disease risk for a new patient.
    
    Parameters:
    -----------
    patient_data : dict
        Dictionary with clinical measurements
    model : sklearn model
        Trained classification model
    
    Returns:
    --------
    dict : Prediction result with risk level and probability
    """
    # Build patient feature vector
    row = pd.DataFrame([patient_data])
    
    # Add engineered features
    row['age_group'] = pd.cut(row['age'], bins=[0, 40, 50, 60, 100],
                               labels=[0, 1, 2, 3]).astype(int)
    row['hr_reserve'] = row['thalach'] - (220 - row['age'])
    row['has_chest_pain'] = (row['cp'] > 0).astype(int)
    
    # Scale features
    row_scaled = scaler.transform(row[feature_names])
    
    # Predict
    pred = model.predict(row_scaled)[0]
    prob = model.predict_proba(row_scaled)[0][1]
    
    risk_level = (
        '🟢 LOW'     if prob < 0.30 else
        '🟡 MODERATE' if prob < 0.60 else
        '🔴 HIGH'
    )
    
    return {
        'prediction': 'Heart Disease' if pred == 1 else 'No Heart Disease',
        'probability': f'{prob*100:.1f}%',
        'risk_level': risk_level
    }


# ─── Test Case 1: High-risk patient ───────────────────────────────────────────
high_risk_patient = {
    'age': 63, 'sex': 1, 'cp': 0, 'trestbps': 145, 'chol': 233,
    'fbs': 1, 'restecg': 0, 'thalach': 150, 'exang': 0,
    'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1
}

# ─── Test Case 2: Low-risk patient ────────────────────────────────────────────
low_risk_patient = {
    'age': 35, 'sex': 0, 'cp': 1, 'trestbps': 115, 'chol': 190,
    'fbs': 0, 'restecg': 1, 'thalach': 182, 'exang': 0,
    'oldpeak': 0.0, 'slope': 2, 'ca': 0, 'thal': 2
}

print('=' * 50)
print('  HEART DISEASE RISK PREDICTION DEMO')
print('=' * 50)

for label, patient in [('High-Risk Patient (63M)', high_risk_patient),
                         ('Low-Risk Patient (35F)',  low_risk_patient)]:
    result = predict_heart_disease(patient)
    print(f'\n👤 {label}')
    print(f'   Prediction  : {result["prediction"]}')
    print(f'   Probability : {result["probability"]}')
    print(f'   Risk Level  : {result["risk_level"]}')

---
# 📝 Step 10: Final Summary & Insights

In [ ]:
# Final radar/spider chart for holistic model comparison
from matplotlib.patches import FancyArrowPatch

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
N = len(metrics_names)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

colors_radar = ['#3498db', '#e67e22', '#9b59b6']

for (name, pred, prob, _), color in zip(models_info, colors_radar):
    values = [
        accuracy_score(y_test, pred),
        precision_score(y_test, pred),
        recall_score(y_test, pred),
        f1_score(y_test, pred),
        roc_auc_score(y_test, prob)
    ]
    values += values[:1]  # close the polygon
    ax.plot(angles, values, 'o-', lw=2, color=color, label=name)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), metrics_names, fontsize=12)
ax.set_ylim(0.5, 1.0)
ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_yticklabels(['0.6', '0.7', '0.8', '0.9', '1.0'], fontsize=9)
ax.set_title('Radar Chart — Model Performance Comparison', fontsize=14,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
print('''
╔══════════════════════════════════════════════════════════════════════╗
║          HEART DISEASE PREDICTION — PROJECT SUMMARY                 ║
╠══════════════════════════════════════════════════════════════════════╣
║  J.N.N College of Engineering, Shivamogga | ML (BCS602) | Sem 6    ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  PROBLEM TYPE : Binary Classification                                ║
║  DATASET      : Cleveland Heart Disease (UCI ML Repository)          ║
║  SAMPLES      : 303 patients | 13 clinical features                  ║
║                                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  WEEK 1-2: DATA EXPLORATION & PREPROCESSING                         ║
║   • Performed thorough EDA with 8+ visualizations                   ║
║   • Handled missing values (median imputation)                       ║
║   • Treated outliers using Winsorization (IQR method)                ║
║   • Created 3 new engineered features                                ║
║   • Applied StandardScaler for feature normalization                 ║
║   • Stratified 80/20 train-test split                                ║
╠══════════════════════════════════════════════════════════════════════╣
║  WEEK 3: MODEL TRAINING (3 Models + Hyperparameter Tuning)          ║
║   ① Logistic Regression  — Baseline interpretable model             ║
║   ② Random Forest         — Best ensemble model (RECOMMENDED)       ║
║   ③ XGBoost               — State-of-the-art gradient boosting      ║
║   All tuned with 5-Fold Stratified Cross-Validation + GridSearchCV  ║
╠══════════════════════════════════════════════════════════════════════╣
║  WEEK 4: EVALUATION & INSIGHTS                                      ║
║   Metrics: Accuracy, Precision, Recall, F1, ROC-AUC                 ║
║   Tools:   Confusion Matrix, ROC Curves, SHAP Explainability         ║
╠══════════════════════════════════════════════════════════════════════╣
║  KEY CLINICAL FINDINGS:                                              ║
║   • thal (Thalassemia type) is the strongest predictor               ║
║   • cp (Chest pain type) — higher types indicate less disease        ║
║   • ca (Fluoroscopy vessels) — more vessels = higher risk            ║
║   • oldpeak (ST depression) — higher values increase risk            ║
║   • thalach (Max heart rate) — lower max HR = higher risk            ║
╠══════════════════════════════════════════════════════════════════════╣
║  POSSIBLE IMPROVEMENTS:                                              ║
║   • Use larger datasets (Framingham Heart Study ~5000+ patients)     ║
║   • Apply SMOTE for class imbalance (if needed)                      ║
║   • Try Neural Networks / Deep Learning                              ║
║   • Deploy as a web application (Flask/FastAPI)                      ║
║   • Integrate with EHR systems for real-time predictions             ║
╚══════════════════════════════════════════════════════════════════════╝
''')

---

## 📚 References

1. Dua, D. and Graff, C. (2019). *UCI Machine Learning Repository* [http://archive.ics.uci.edu/ml]. Irvine, CA: University of California, School of Information and Computer Science.
2. Detrano, R., et al. (1989). *International application of a new probability algorithm for the diagnosis of coronary artery disease.* American Journal of Cardiology, 64(5), 304-310.
3. Breiman, L. (2001). *Random Forests.* Machine Learning, 45(1), 5–32.
4. Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16.
5. Lundberg, S. M., & Lee, S. I. (2017). *A Unified Approach to Interpreting Model Predictions.* NeurIPS.
6. World Health Organization. (2023). *Cardiovascular diseases (CVDs).* WHO Fact Sheet.

---

*Submitted by: [Your Name] | Roll No: [Your Roll No] | JNNCE Shivamogga | 2024-25*